In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm
import shutil
import json
import pickle
import time

try:
    import xmltodict
except:
    ! pip install xmltodict

# binning
try:
    from optbinning import OptimalBinning
except:
    ! pip install optbinning

from preprocessing import Preprocessing
from api import ParsePayload

#### Functions

#### Constants

In [2]:
str_splitter = '/'

# project
str_project = os.getcwd().split(str_splitter)[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split(str_splitter)[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split(str_splitter)[6]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

# tiers
str_tiers = """
{
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500,
}
"""
str_tiers = str_tiers.replace(' ','')

# pricing
flt_pct_threshold = 0.10
int_dollars_round_fees = 1
flt_avg_life = 2.3
flt_equity_intercept = 0.04
flt_equity_slope = 0.80
flt_securitization = 0.0595
flt_late_fee_income = 0.0042
flt_state_rate_cap = 1.0
flt_cnl_scaler = 0.8039
flt_prop_c = 0.5

Project: 20241112-simple-model-test
Task: 13_60_in_720_dl
Subtask: 06_parser


#### Make output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Read payload

In [4]:
str_filename = 'request_8473752_12.json'
str_local_path = f'./input/{str_filename}'
dict_json_request = json.load(open(str_local_path))['request']
str_json_request = json.dumps(dict_json_request)

#### Copy preprocessing script

In [5]:
str_filename = 'preprocessing.py'
str_origin = f'../../08_prep_data/{str_filename}'
str_destination = f'./{str_filename}'
shutil.copyfile(str_origin, str_destination)

'./preprocessing.py'

#### Get preprocessing model and attributes

In [6]:
# import
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'../../08_prep_data/output/{str_filename}'
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

# get attributes
flt_quantile = cls_model_preprocessing.flt_quantile
flt_income_max = cls_model_preprocessing.flt_income_max

# get imputation dictionary
str_filename = 'dict_impute.pkl'
str_local_path = f'../../14_60_in_720_dl_test/02_model/output/{str_filename}'
dict_impute = pickle.load(open(str_local_path, 'rb'))

# get bins for scorecard
str_filename = 'dict_bins.pkl'
str_local_path = f'../../14_60_in_720_dl_test/02_model/output/{str_filename}'
dict_bins = pickle.load(open(str_local_path, 'rb'))

#### Initialize class

In [7]:
# init
cls_model_preprocessing = Preprocessing(
    dict_impute=dict_impute,
    dict_bins=dict_bins,
    int_new_payment=0,
)
# assign
cls_model_preprocessing.flt_income_max = flt_income_max

# save
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_model_preprocessing, open(str_local_path, 'wb'))

#### Get inference model

In [8]:
str_filename = 'cls_model_inference_ml_logistic_scorecard.pkl'
str_local_path = f'../02_model/output/{str_filename}'
cls_model_inference = pickle.load(open(str_local_path, 'rb'))

#### Get list of features

In [9]:
list_cols_model = list(cls_model_inference.feature_names_in_)
list_cols_model = [f'{col.split("_binned")[0]}' for col in list_cols_model]
print(f'There are {len(list_cols_model)} features in the model')

There are 33 features in the model


#### Columns to force

In [10]:
list_cols_force = [
    'applicationdate__app',
    'fltgrossmonthly__income_sum',
    'amtfinanced__app',
    'bookvalue__app',
    'int_n_months_open__tu_pmthx',
    'int_n_months_closed__tu_pmthx',
    'int_n_months_open__tu_pmthx',
    'int_n_months_closed__tu_pmthx',
    'flt_wtd_avg_open__tu_pmthx',
    'flt_wtd_avg_closed__tu_pmthx',
    'int_bad_3mo_open__tu_pmthx',
    'int_bad_3mo_closed__tu_pmthx',
    'int_bad_6mo_open__tu_pmthx',
    'int_bad_6mo_closed__tu_pmthx',
    'int_bad_3mo_open_end__tu_pmthx',
    'int_bad_3mo_closed_end__tu_pmthx',
    'int_bad_6mo_open_end__tu_pmthx',
    'int_bad_6mo_closed_end__tu_pmthx',
    'strdealershiptrackertype__app',
    'bitdebtor__app',
    'bigaccountid__app',
    'vehicleyear__app',
    'flt_payment_open__tu_pmthx',
    'intopenbktype__app',
    'flt_avg_open__tu_pmthx',
    'flt_avg_closed__tu_pmthx',
    'totaldebt__app',
]
print(f'There are {len(list_cols_force)} columns needed for preprocessing')

There are 27 columns needed for preprocessing


#### Combine lists

In [11]:
list_cols_necessary = list_cols_model + list_cols_force
# rm dups
list_cols_necessary = list(dict.fromkeys(list_cols_necessary))
print(f'There are {len(list_cols_necessary)} columns necessary for preprocessing and prediction')

There are 54 columns necessary for preprocessing and prediction


#### Adverse action dictionary

In [12]:
df_aa = pd.read_csv('./input/df_aa.csv')
dict_aa = dict(zip(df_aa['feature'], df_aa['reason']))

# ensure captialized
dict_aa = {key: val.capitalize() for key, val in dict_aa.items()}

# pickle
str_filename = 'dict_aa.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(dict_aa, open(str_local_path, 'wb'))

#### Make sure there is an AA reason for every feature in the model

In [13]:
list_cols_model = list(cls_model_inference.feature_names_in_)
list_cols_model = [col.split('_binned')[0] for col in list_cols_model]
for a, col in enumerate(list_cols_model):
    try:
        print(f'{a+1} - {col}: {dict_aa[col]}')
    except:
        print(f'{a+1} - {col}: ERROR')
        dict_aa[col] = 'PLACEHOLDER'

1 - rev326__tu: Excessive account balances
2 - au20s__tu: Insufficient credit file, length of credit
3 - linkt001__tu: Length of residence
4 - miles_odometer__app: Value or type of collateral not sufficient
5 - balmag01__tu: Excessive account balances
6 - fltgrossmonthly__income_sum: Insufficient income
7 - br109s__tu: Delinquent credit obligations
8 - int_n_months_open__tu_pmthx: Insufficient credit history
9 - rtl_trd__tu: Insufficient credit file, length of credit
10 - ENG-loan_to_value: Value or type of collateral not sufficient
11 - trv10__tu: Excessive account balances
12 - trv13__tu: High revolving credit balances
13 - cv15__tu: Excessive inquiries
14 - rp01s__tu: Garnishment, or attachment, or foreclosure, or repossession, or suit
15 - jt20s__tu: Insufficient length of credit history 
16 - linkt003__tu: Length of residence
17 - g002s__tu: Delinquent credit obligations
18 - g407s__tu: Excessive inquiries
19 - g232s__tu: Excessive inquiries
20 - flt_wtd_avg_closed__tu_pmthx: Dero

#### Get the LTV bins

In [14]:
# open
str_filename = 'dict_bins_ltv.pkl'
str_local_path = f'../../create_grid/04_60_in_720_chargeoff_dl/output/{str_filename}'
dict_bins_ltv = pickle.load(open(str_local_path, 'rb'))

#### Initialize class

In [15]:
cls_parser = ParsePayload(
    cls_model_preprocessing=cls_model_preprocessing,
    cls_model_inference=cls_model_inference,
    dict_aa=dict_aa,
    str_tiers=str_tiers,
    list_cols_necessary=list_cols_necessary,
    dict_bins_ltv=dict_bins_ltv,
)

#### Start time

In [16]:
time_start = time.perf_counter()

#### Parse payload

In [17]:
# get data
cls_parser.get_data(str_request=str_json_request)
# engineer pmt hx
cls_parser.engineer_pmt_hx()
# preprocessing
cls_parser.preprocessing()
# get predictions
cls_parser.get_predictions()
# interpolate
#cls_parser.interpolate()
# adverse action
cls_parser.adverse_action()
# counter offers
#cls_parser.counter_offers()
# generate response
cls_parser.generate_response()

Getting data...
['uniqueid__app', 'uniqueid__ln']
[8473752104567021]: Engineering payment history...


100%|██████████| 29/29 [00:00<00:00, 2322.92it/s]


Chime: True
[8473752104567021]: Preprocessing data...


100%|██████████| 54/54 [00:00<00:00, 5266.53it/s]


Masking negative values to NaN...


...

Binning values for scorecard...


100%|██████████| 982/982 [00:00<00:00, 24416.42it/s]


[8473752104567021]: Getting predictions...
[8473752104567021]: Getting adverse action...


100%|██████████| 33/33 [00:00<00:00, 4717.36it/s]
1it [00:00, 2532.79it/s]
100%|██████████| 1/1 [00:00<00:00, 4981.36it/s]

[8473752104567021]: Generating response...


#### End time

In [18]:
time_end = time.perf_counter()
flt_sec = time_end - time_start
print(f'Time to parse: {flt_sec:0.4f} sec.')

Time to parse: 0.3384 sec.


#### Show response

In [19]:
cls_parser.dict_response

{'Request_id': '',
 'Zaml_processing_id': '',
 'Response': [{'Model_name': 'prestige-dlv2',
   'Model_version': 'v1',
   'Results': [{'Row_id': 8473752104567021,
     'Score_pd': 0.1312261427,
     'Score_lgd': 0.6356838066,
     'Score_ecnl': 0.0834183339,
     'Score_ecnl_mod': 0.2368313235,
     'Key_factors': ['Garnishment, or attachment, or foreclosure, or repossession, or suit',
      'Value or type of collateral not sufficient',
      'Delinquent credit obligations',
      'Delinquent credit obligations',
      'Insufficient credit history'],
     'Outlier_score': 0.0,
     'Dict_tiers': "\n{\n'A1':0.0760,\n'A':0.1320,\n'B':0.2650,\n'C':0.3220,\n'D':0.3500,\n}\n"}],
   'Errors': [],
   'CounterOffers': []}]}

#### Save

In [20]:
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_parser, open(str_local_path, 'wb'))